In [24]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

In [25]:
df=pd.read_csv('../data/healthcare_cleaned.csv')

In [26]:
df.isna().sum().sum()

np.int64(0)

In [27]:
drop_cols=['patient_id','treatment_cost','mortality_risk','diagnosis_category','length_of_stay','readmission_30day']

In [28]:
X=df.drop(drop_cols,axis=1)
y=df['readmission_30day']

In [29]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
cat_cols=['gender', 'smoking_status', 'admission_type','department', 'insurance_type']
num_cols=[col for col in X.columns if col not in cat_cols]
Preprocessor=ColumnTransformer(transformers=[
    ('num',StandardScaler(),num_cols),
    ('cat',OneHotEncoder(handle_unknown='ignore'),cat_cols)
])
pipeline=Pipeline(steps=[
    ('preprocessor',Preprocessor),
    ('model',DecisionTreeClassifier(class_weight='balanced',random_state=42))
])

In [30]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
pipeline.fit(X_train,y_train)
y_pred=pipeline.predict(X_test)

In [31]:
from sklearn.metrics import classification_report,confusion_matrix,f1_score,accuracy_score
print(classification_report(y_test,y_pred))
print(confusion_matrix(y_pred,y_test))
print(accuracy_score(y_pred,y_test))

              precision    recall  f1-score   support

           0       0.85      0.85      0.85      8515
           1       0.15      0.15      0.15      1485

    accuracy                           0.75     10000
   macro avg       0.50      0.50      0.50     10000
weighted avg       0.75      0.75      0.75     10000

[[7247 1263]
 [1268  222]]
0.7469


In [32]:
params={
    'model__max_depth':[3,5,7],
    'model__min_samples_split':[5,10,20,50,100]
}
from sklearn.model_selection import GridSearchCV
grid_model=GridSearchCV(cv=5,param_grid=params,n_jobs=-1,estimator=pipeline,scoring='f1')
grid_model.fit(X_train,y_train)
y_pred=grid_model.predict(X_test)

In [33]:
print(classification_report(y_pred,y_test))

              precision    recall  f1-score   support

           0       0.65      0.87      0.74      6302
           1       0.47      0.19      0.27      3698

    accuracy                           0.62     10000
   macro avg       0.56      0.53      0.51     10000
weighted avg       0.58      0.62      0.57     10000



In [34]:
print(grid_model.best_params_)
print(grid_model.best_score_)

{'model__max_depth': 3, 'model__min_samples_split': 5}
0.2726155621760435


Observations:
1) DT has f1_Score of 0.27 as best at max_Depth=3 and min_Samples_split=5 of minority class
2)  Recall is o.19, nearly missing out of 81% of actual readmission values
3) Hence it deep tree overfits, shallow goes on underfit
4) need to ensemble methods such as Random Forest , for scope of improvement

In [35]:
pipeline_RF=Pipeline(steps=[
    ('preprocessor',Preprocessor),
    ('model_rf',RandomForestClassifier(class_weight='balanced',random_state=42))
])
pipeline_RF.fit(X_train,y_train)
y_pred=pipeline_RF.predict(X_test)
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.85      1.00      0.92      8515
           1       0.00      0.00      0.00      1485

    accuracy                           0.85     10000
   macro avg       0.43      0.50      0.46     10000
weighted avg       0.73      0.85      0.78     10000



c:\Users\admin\Health_Care\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\admin\Health_Care\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\admin\Health_Care\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [36]:
params_rf={
    'model_rf__n_estimators':[10,50,100,200],
    'model_rf__max_depth':[3,5,7],
    'model_rf__min_samples_split':[3,5,7,10]
}
grid_model_rf=GridSearchCV(estimator=pipeline_RF,cv=5,scoring='f1',n_jobs=-1,param_grid=params_rf)
grid_model_rf.fit(X_train,y_train)
y_pred=grid_model_rf.predict(X_test)
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.88      0.56      0.68      8515
           1       0.18      0.56      0.27      1485

    accuracy                           0.56     10000
   macro avg       0.53      0.56      0.48     10000
weighted avg       0.78      0.56      0.62     10000



In [37]:
print(grid_model_rf.best_params_)
print(grid_model_rf.best_score_)

{'model_rf__max_depth': 5, 'model_rf__min_samples_split': 5, 'model_rf__n_estimators': 50}
0.2770977045924778


Observations:
1) RF has f1_Score of 0 , nothing is predicted without tuning
2)  Tuned RF :f1 is 0.27, nearly missing out of 73% of actual readmission values at max_depth as 5 and estimators as 50, marginal improvement could be seen in f1, good improvement on recall
3) need to check on XGBOOST , for scope of improvement

In [38]:
scale_pos_weight=y_train.value_counts()[0]/y_train.value_counts()[1]
#around 5.65
pipeline_xgb=Pipeline(steps=[
    ('preprocessor',Preprocessor),
    ('model_xgb',XGBClassifier(scale_pos_weight=scale_pos_weight,eval_metric='logloss'))
])
pipeline_xgb.fit(X_train,y_train)
y_pred=pipeline_xgb.predict(X_test)
print(classification_report(y_test,y_pred))
params_xgb={
    'model_xgb__n_estimators':[10,50,100,200],
    'model_xgb__max_depth':[3,5,7],
    'model_xgb__learning_rate':[0.01,0.1,0.3]
}
grid_model_xgb=GridSearchCV(cv=5,estimator=pipeline_xgb,n_jobs=-1,param_grid=params_xgb,scoring='f1')
grid_model_xgb.fit(X_train,y_train)
y_pred=grid_model_xgb.predict(X_test)
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.86      0.76      0.81      8515
           1       0.16      0.27      0.20      1485

    accuracy                           0.69     10000
   macro avg       0.51      0.52      0.51     10000
weighted avg       0.75      0.69      0.72     10000

              precision    recall  f1-score   support

           0       0.88      0.51      0.65      8515
           1       0.18      0.60      0.27      1485

    accuracy                           0.53     10000
   macro avg       0.53      0.56      0.46     10000
weighted avg       0.78      0.53      0.59     10000



In [39]:
print(grid_model_xgb.best_params_)
print(grid_model_xgb.best_score_)

{'model_xgb__learning_rate': 0.1, 'model_xgb__max_depth': 3, 'model_xgb__n_estimators': 10}
0.27603397518559813


observations:
1 ) even xgb failed, this leads us to conclude ; due to weak signals and not much impacted features , failed to provide enough insights to predict readmission, as far as imbalance is considered we tried penalize using balance param value , now we try to use resampling tech such as SMOTE

In [40]:
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
smote_pipeline=ImbPipeline(steps=[
    ('preprocessor',Preprocessor),
    ('smote',SMOTE(random_state=42)),
    ('model_xgb',XGBClassifier(random_state=42,eval_metric='logloss'))
])
smote_pipeline.fit(X_train,y_train)
y_pred=smote_pipeline.predict(X_test)
print(classification_report(y_test,y_pred))
params_xgb={
    'model_xgb__n_estimators':[10,50,100,200],
    'model_xgb__max_depth':[3,5,7],
    'model_xgb__learning_rate':[0.01,0.1,0.3]
}
grid_model_xgb=GridSearchCV(cv=5,estimator=smote_pipeline,n_jobs=-1,param_grid=params_xgb,scoring='f1')
grid_model_xgb.fit(X_train,y_train)
y_pred=grid_model_xgb.predict(X_test)
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.85      1.00      0.92      8515
           1       0.23      0.01      0.01      1485

    accuracy                           0.85     10000
   macro avg       0.54      0.50      0.47     10000
weighted avg       0.76      0.85      0.78     10000

              precision    recall  f1-score   support

           0       0.86      0.84      0.85      8515
           1       0.19      0.22      0.20      1485

    accuracy                           0.75     10000
   macro avg       0.53      0.53      0.53     10000
weighted avg       0.76      0.75      0.75     10000



In [41]:
print(grid_model_xgb.best_params_)
print(grid_model_xgb.best_score_)

{'model_xgb__learning_rate': 0.01, 'model_xgb__max_depth': 3, 'model_xgb__n_estimators': 10}
0.20296788527417636


class_weight outperformed SMOTE on this dataset. SMOTE didn't help because the problem is weak feature signal, not just imbalance. Creating synthetic samples of weak features doesn't add information.

Winner: RF + class_weight

In [42]:
y_pred_proba = grid_model_rf.predict_proba(X_test)[:,1]
y_pred_threshold = (y_pred_proba > 0.3).astype(int)
print(classification_report(y_test, y_pred_threshold))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00      8515
           1       0.15      1.00      0.26      1485

    accuracy                           0.15     10000
   macro avg       0.07      0.50      0.13     10000
weighted avg       0.02      0.15      0.04     10000



c:\Users\admin\Health_Care\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\admin\Health_Care\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\admin\Health_Care\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [43]:
y_pred_threshold = (y_pred_proba > 0.4).astype(int)
print(classification_report(y_test, y_pred_threshold))

              precision    recall  f1-score   support

           0       0.92      0.12      0.21      8515
           1       0.16      0.94      0.27      1485

    accuracy                           0.24     10000
   macro avg       0.54      0.53      0.24     10000
weighted avg       0.80      0.24      0.22     10000



In [44]:
y_pred_threshold = (y_pred_proba > 0.45).astype(int)
print(classification_report(y_test, y_pred_threshold))

              precision    recall  f1-score   support

           0       0.89      0.23      0.37      8515
           1       0.16      0.83      0.27      1485

    accuracy                           0.32     10000
   macro avg       0.52      0.53      0.32     10000
weighted avg       0.78      0.32      0.36     10000



In [46]:
import joblib
joblib.dump(grid_model_rf, '../models/readmission_model.pkl')

['../models/readmission_model.pkl']

Best model: RF + class_weight

Best threshold: 0.4

Minority recall: 0.94, F1: 0.27

Conclusion: weak feature signal limits F1, but high recall means most readmissions are caught